# 08. Relative claim status and canonical receipts

**Level:** Advanced

**Prerequisites:** Lessons 020 and 070; familiarity with the distinction between evidence and claim status.

**Canonical claim anchors:** `WDP-CH20-D01`, `WDP-CH20-O01`, `WDP-CH20-N01`

## Learning objectives

- route a claim through an explicit evaluator profile
- attach a finite witness without promoting source status
- bind inputs and outputs to a canonical receipt and detect drift

Work top-to-bottom. Predict each worked result before executing it, then use
the exercise to test whether you can separate the finite result from its scope
boundary.


In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
root = next((candidate for candidate in (start, *start.parents)
             if (candidate / 'pyproject.toml').is_file()
             and (candidate / 'code' / 'witnessed_difference').is_dir()), None)
if root is None:
    raise RuntimeError('could not locate the self-contained paper package')
sys.path.insert(0, str(root / 'code'))


## Idea: evidence attachment is not status promotion

A profile declares which claim identifiers it admits. Routing preserves a
claim's source status when admitted and returns `OUTSIDE_PROFILE` otherwise.
Attaching a finite witness records evidence but leaves open, conjectural,
interpretive, or refuted status unchanged.

Canonical receipts complement this rule by binding a subject, version, sorted
claim IDs, input digest, and output digest. They detect artifact drift; they do
not decide whether the source claim deserved its status.


## Visual checkpoint

This lesson uses printed exact data rather than a diagram. Read each returned
status and refusal as part of the result; do not reduce the cell to its final
number.


## Derivation

Receipt construction canonicalizes supported data before hashing. Claim IDs
are deduplicated and sorted, so their input order cannot alter the receipt.
Verification recomputes the receipt body, then separately checks the supplied
input and output artifacts. These three checks distinguish body mutation,
input mutation, and output mutation.


In [ ]:
from witnessed_difference.notebook_support import display_json
from witnessed_difference.relative_status import Claim, ClaimStatus, EvaluationProfile, attach_finite_witness, route_claim

profile = EvaluationProfile('finite-review', frozenset({'WDP-CH20-O01'}))
source = Claim('WDP-CH20-O01', ClaimStatus.FINITE_PROPOSITION)
routed = route_claim(source, profile)
outside = route_claim(Claim('WDP-CH20-N01', ClaimStatus.NONCLAIM), profile)
print(display_json({'routed': routed, 'with_witness': attach_finite_witness(routed, 'loop-1'), 'outside': outside}))


## Read the result

The finite routing witness preserves the written proposition's mathematical
status; it does not certify its formalization or paired reproduction. The
nonclaim outside the profile is not labelled false; it remains explicitly
outside this evaluator's admitted domain. This avoids treating silence as
refutation or success.


In [ ]:
from witnessed_difference.receipts import make_receipt, verify_receipt

inputs = {'connection': 'triangle', 'path': ['a', 'b', 'c', 'a']}
output = {'holonomy': '-1'}
receipt = make_receipt('declared-loop', inputs, output, ('WDP-CH05-D01', 'WDP-CH05-D01')).value
print(display_json({'receipt': receipt, 'verified': verify_receipt(receipt, inputs, output)}))
print(display_json({'mutated_output': verify_receipt(receipt, inputs, {'holonomy': '+1'})}))


## Exercise

Reverse the order of keys in the input mapping and verify the receipt again.
Then add one new key. Explain why canonical key ordering accepts the first
change while content binding rejects the second.


In [ ]:
reordered = {'path': ['a', 'b', 'c', 'a'], 'connection': 'triangle'}
extended = {'path': ['a', 'b', 'c', 'a'], 'connection': 'triangle', 'budget': 3}
print(display_json({'reordered': verify_receipt(receipt, reordered, output), 'extended': verify_receipt(receipt, extended, output)}))


## Scope check

Receipts establish content identity under a declared serialization.
They do not establish external truth, authorship, review, or authorization.
**AuthorityEffect: None.**


## Continue the learning path

- Previous: [Declared U(1) phases, holonomy, and gauge change](070_declared_u1_holonomy.ipynb)
- Index: [Notebook curriculum](README.md)
- Next: [Mutation recovery, evidence composition, and terminal scope](090_mutation_recovery_and_scope.ipynb)
